# Experiment 5: Decision Tree and Random Forest - A Comparative Classification Study

```
experiment5_dt_rf_breast_cancer.py
====================================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 5: Decision Tree and Random Forest - A Comparative Classification Study

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                         -> generate_eda_summary()
    - Classification train/eval   -> train_evaluate_classification()
    - Classification metrics      -> classification_performance_metrics()
    - Global plot style           -> set_plot_style()

Dataset: Wisconsin Diagnostic Breast Cancer (WDBC), 569 samples, 30 numeric
features, binary target (Malignant / Benign). Loaded from a CSV file
(breast_cancer_data.csv) in the standard Kaggle WDBC format: an "id"
column, a "diagnosis" column ('M' = malignant, 'B' = benign), and 30 named
feature columns (mean/error/worst radius, texture, perimeter, ...).

NOTE on label encoding: this script encodes diagnosis 'M' (malignant) as
target = 0 and 'B' (benign) as target = 1, matching the convention used by
sklearn.datasets.load_breast_cancer (i.e. "0" is the more serious/
positive-for-disease class). This is the reverse of the 0=negative/
1=positive convention many students expect, so it is called out explicitly
here and in the report to avoid misreading the confusion matrix.
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_validate, GridSearchCV)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET AND ENCODE CLASS LABELS

In [2]:
DATA_PATH = "breast_cancer_data.csv"  # <-- upload this file to the notebook's working directory
raw = pd.read_csv(DATA_PATH)

# Drop non-feature identifier columns if present (Kaggle WDBC CSVs sometimes
# include an "id" column and/or a trailing unnamed column).
drop_cols = [c for c in raw.columns if c.lower() == "id" or "unnamed" in c.lower()]
df = raw.drop(columns=drop_cols).copy()

feature_names = [c for c in df.columns if c != "diagnosis"]

# target: 0 = malignant (M), 1 = benign (B)
df["target"] = df["diagnosis"].map({"M": 0, "B": 1})
df["diagnosis"] = df["diagnosis"].map({"M": "Malignant", "B": "Benign"})

print("Dataset shape:", df.shape)
print(df["diagnosis"].value_counts())
print("Missing values:", int(df[feature_names].isnull().sum().sum()))

Dataset shape: (569, 32)
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64
Missing values: 0


## 2. EDA (reusable function from Experiment 1)

In [3]:
eda_df = df.drop(columns=["target"])
generate_eda_summary(
    eda_df, target_col="diagnosis", dataset_name="Wisconsin Breast Cancer",
    save_path=f"{FIG_DIR}/eda_breast_cancer.eps"
)
plt.close("all")

# Feature correlation heatmap (full 30x30, required explicitly by manual)
fig, ax = plt.subplots(figsize=(14, 12))
corr = df[feature_names].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar=True,
            xticklabels=True, yticklabels=True)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
_bold_axis_labels(ax, title="Full Feature Correlation Heatmap (30 features)")
_save_eps(fig, f"{FIG_DIR}/full_correlation_heatmap.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_breast_cancer.eps (600 DPI, EPS)


## 3. TRAIN / TEST SPLIT (80-20, stratified)

In [4]:
X = df[feature_names].values
y = df["target"].values  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def time_fit_predict(model, Xtr, Xte, ytr):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_pred = model.predict(Xte)
    pred_t = time.perf_counter() - t0
    return y_pred, train_t, pred_t

## 4. BASELINE DECISION TREE (via reusable train_evaluate_classification)

In [5]:
baseline_models = {"Decision Tree (Baseline)": DecisionTreeClassifier(random_state=RANDOM_STATE)}
baseline_results_df, baseline_fitted = train_evaluate_classification(
    baseline_models, X_train, X_test, y_train, y_test, scale=False
)
print("\n=== Baseline Decision Tree (default hyperparameters) ===")
print(baseline_results_df)

--- Classification Metrics: Decision Tree (Baseline) ---
  Accuracy  : 0.9123
  Precision : 0.9161
  Recall    : 0.9123
  F1-score  : 0.9130
  ROC-AUC   : 0.9157


=== Baseline Decision Tree (default hyperparameters) ===
                          Accuracy  Precision    Recall  F1-score   ROC-AUC
Model                                                                      
Decision Tree (Baseline)  0.912281   0.916072  0.912281  0.913021  0.915675


## 5. DECISION TREE HYPERPARAMETER SEARCH SPACE + 5-FOLD CV EVALUATION

In [6]:
dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

print("\n[Decision Tree] Starting GridSearchCV (5-fold)...")
t0 = time.perf_counter()
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), dt_param_grid,
                        cv=cv_strategy, scoring="accuracy", n_jobs=1)
dt_grid.fit(X_train, y_train)
dt_grid_time = time.perf_counter() - t0
print(f"[Decision Tree] GridSearchCV done in {dt_grid_time:.1f}s")
print("Best params:", dt_grid.best_params_)
print("Best CV accuracy:", dt_grid.best_score_)

# Table 1: Criterion x Max Depth summary (averaging over the other params via
# best-per-combination selection, as requested by the manual's table shape)
dt_cv_results = pd.DataFrame(dt_grid.cv_results_)
dt_table1_rows = []
for criterion in ["gini", "entropy"]:
    for max_depth in [3, 5, 7, 10, None]:
        if max_depth is None:
            mask = dt_cv_results["param_max_depth"].isna()
        else:
            mask = dt_cv_results["param_max_depth"] == max_depth
        subset = dt_cv_results[(dt_cv_results["param_criterion"] == criterion) & mask]
        if subset.empty:
            continue
        best_row = subset.loc[subset["mean_test_score"].idxmax()]
        # Compute F1 for this specific best combination via a fresh 5-fold CV
        params = best_row["params"]
        model = DecisionTreeClassifier(random_state=RANDOM_STATE, **params)
        cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                                 scoring=["accuracy", "f1"])
        dt_table1_rows.append({
            "Criterion": criterion,
            "Max Depth": "None" if max_depth is None else max_depth,
            "Avg CV Accuracy (%)": cv_res["test_accuracy"].mean() * 100,
            "Avg CV F1 Score": cv_res["test_f1"].mean(),
        })
dt_table1_df = pd.DataFrame(dt_table1_rows)
dt_table1_df.to_csv(f"{RES_DIR}/dt_hyperparameter_evaluation.csv", index=False)
print("\n=== Table 1: Decision Tree Hyperparameter Evaluation (5-Fold CV) ===")
print(dt_table1_df)

best_dt = dt_grid.best_estimator_
y_pred_dt, dt_train_t, dt_pred_t = time_fit_predict(best_dt, X_train, X_test, y_train)
y_proba_dt = best_dt.predict_proba(X_test)
dt_metrics = classification_performance_metrics(
    y_test, y_pred_dt, y_proba_dt, model_name="Decision Tree (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_decision_tree.eps"
)
dt_metrics["Training Time (s)"] = dt_train_t
print("\n=== Tuned Decision Tree Performance ===")
print(dt_metrics)


[Decision Tree] Starting GridSearchCV (5-fold)...


[Decision Tree] GridSearchCV done in 2.4s
Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV accuracy: 0.9384615384615385



=== Table 1: Decision Tree Hyperparameter Evaluation (5-Fold CV) ===
  Criterion Max Depth  Avg CV Accuracy (%)  Avg CV F1 Score
0      gini         3            93.186813         0.945548
1      gini         5            93.846154         0.950580
2      gini         7            93.846154         0.950580
3      gini        10            93.846154         0.950580
4      gini      None            93.846154         0.950580
5   entropy         3            93.406593         0.947973
6   entropy         5            92.967033         0.943867
7   entropy         7            92.967033         0.943450
8   entropy        10            92.967033         0.943450
9   entropy      None            92.967033         0.943450
--- Classification Metrics: Decision Tree (Tuned) ---
  Accuracy  : 0.9035
  Precision : 0.9061
  Recall    : 0.9035
  F1-score  : 0.9041
  ROC-AUC   : 0.9358


=== Tuned Decision Tree Performance ===
{'Model': 'Decision Tree (Tuned)', 'Accuracy': 0.9035087719298246, 'P

## 6. OVERFITTING ANALYSIS: TREE DEPTH vs TRAIN/VALIDATION ACCURACY

In [7]:
depths = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 20, None]
depth_rows = []
for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                             scoring="accuracy", return_train_score=True)
    depth_rows.append({
        "Max Depth": "None" if d is None else d,
        "Train Accuracy": cv_res["train_score"].mean(),
        "Validation Accuracy": cv_res["test_score"].mean(),
    })
depth_df = pd.DataFrame(depth_rows)
depth_df.to_csv(f"{RES_DIR}/depth_vs_accuracy.csv", index=False)
print("\n=== Tree Depth vs Train/Validation Accuracy ===")
print(depth_df)

fig, ax = plt.subplots(figsize=(8, 5))
x_labels = [str(d) for d in depth_df["Max Depth"]]
x_pos = np.arange(len(x_labels))
ax.plot(x_pos, depth_df["Train Accuracy"], marker="o", label="Training Accuracy", color="#2980b9")
ax.plot(x_pos, depth_df["Validation Accuracy"], marker="s", label="Validation Accuracy", color="#c0392b")
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.legend()
_bold_axis_labels(ax, "Max Depth", "Accuracy", "Decision Tree: Overfitting vs Tree Depth")
_save_eps(fig, f"{FIG_DIR}/depth_vs_accuracy.eps")
plt.close(fig)

# Visualize the tuned (best) decision tree structure (top levels only, for readability)
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(best_dt, max_depth=3, feature_names=feature_names,
          class_names=["Malignant", "Benign"], filled=True, fontsize=8, ax=ax)
ax.set_title("Best Decision Tree Structure (first 3 levels shown)",
             fontproperties=fm.FontProperties(family="Times New Roman", weight="bold", size=15))
_save_eps(fig, f"{FIG_DIR}/decision_tree_structure.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Tree Depth vs Train/Validation Accuracy ===
   Max Depth  Train Accuracy  Validation Accuracy
0          1        0.925824             0.903297
1          2        0.961538             0.931868
2          3        0.979121             0.925275
3          4        0.987912             0.929670
4          5        0.992308             0.923077
5          6        0.997253             0.925275
6          7        0.998352             0.923077
7          8        0.999451             0.923077
8         10        1.000000             0.916484
9         12        1.000000             0.916484
10        15        1.000000             0.916484
11        20        1.000000             0.916484
12      None        1.000000             0.916484


## 7. RANDOM FOREST HYPERPARAMETER SEARCH SPACE + 5-FOLD CV EVALUATION

In [8]:
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
}

print("\n[Random Forest] Starting GridSearchCV (5-fold)...")
t0 = time.perf_counter()
rf_grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_param_grid,
                        cv=cv_strategy, scoring="accuracy", n_jobs=1)
rf_grid.fit(X_train, y_train)
rf_grid_time = time.perf_counter() - t0
print(f"[Random Forest] GridSearchCV done in {rf_grid_time:.1f}s")
print("Best params:", rf_grid.best_params_)
print("Best CV accuracy:", rf_grid.best_score_)

# Table 2: n_estimators x Max Depth summary (best over max_features/bootstrap)
rf_cv_results = pd.DataFrame(rf_grid.cv_results_)
rf_table2_rows = []
for n_est in [50, 100, 200]:
    for max_depth in [5, 10, None]:
        if max_depth is None:
            depth_mask = rf_cv_results["param_max_depth"].isna()
        else:
            depth_mask = rf_cv_results["param_max_depth"] == max_depth
        subset = rf_cv_results[(rf_cv_results["param_n_estimators"] == n_est) & depth_mask]
        if subset.empty:
            continue
        best_row = subset.loc[subset["mean_test_score"].idxmax()]
        params = best_row["params"]
        model = RandomForestClassifier(random_state=RANDOM_STATE, **params)
        cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                                 scoring=["accuracy", "f1"])
        rf_table2_rows.append({
            "n_estimators": n_est,
            "Max Depth": "None" if max_depth is None else max_depth,
            "Max Features": params["max_features"],
            "Avg CV Accuracy (%)": cv_res["test_accuracy"].mean() * 100,
            "Avg CV F1 Score": cv_res["test_f1"].mean(),
        })
rf_table2_df = pd.DataFrame(rf_table2_rows)
rf_table2_df.to_csv(f"{RES_DIR}/rf_hyperparameter_evaluation.csv", index=False)
print("\n=== Table 2: Random Forest Hyperparameter Evaluation (5-Fold CV) ===")
print(rf_table2_df)

best_rf = rf_grid.best_estimator_
y_pred_rf, rf_train_t, rf_pred_t = time_fit_predict(best_rf, X_train, X_test, y_train)
y_proba_rf = best_rf.predict_proba(X_test)
rf_metrics = classification_performance_metrics(
    y_test, y_pred_rf, y_proba_rf, model_name="Random Forest (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_random_forest.eps"
)
rf_metrics["Training Time (s)"] = rf_train_t
print("\n=== Tuned Random Forest Performance ===")
print(rf_metrics)


[Random Forest] Starting GridSearchCV (5-fold)...


[Random Forest] GridSearchCV done in 27.1s
Best params: {'bootstrap': True, 'max_depth': 10, 'max_features': 'log2', 'n_estimators': 100}
Best CV accuracy: 0.964835164835165



=== Table 2: Random Forest Hyperparameter Evaluation (5-Fold CV) ===
   n_estimators Max Depth Max Features  Avg CV Accuracy (%)  Avg CV F1 Score
0            50         5         sqrt            96.043956         0.968679
1            50        10         log2            96.263736         0.970243
2            50      None         log2            96.263736         0.970243
3           100         5         sqrt            96.043956         0.968530
4           100        10         log2            96.483516         0.971801
5           100      None         log2            96.483516         0.971801
6           200         5         log2            95.824176         0.966609
7           200        10         sqrt            96.263736         0.970027
8           200      None         sqrt            96.263736         0.970027
--- Classification Metrics: Random Forest (Tuned) ---
  Accuracy  : 0.9561
  Precision : 0.9561
  Recall    : 0.9561
  F1-score  : 0.9560
  ROC-AUC   : 0.9924




=== Tuned Random Forest Performance ===
{'Model': 'Random Forest (Tuned)', 'Accuracy': 0.956140350877193, 'Precision': 0.9560729421281235, 'Recall': 0.956140350877193, 'F1-score': 0.9560273762928302, 'ROC-AUC': 0.9923941798941799, 'Training Time (s)': 0.13825830299992958}


## 8. HYPERPARAMETER TUNING RESULTS SUMMARY TABLE

In [9]:
tuning_summary = pd.DataFrame({
    "Decision Tree": {
        "Search Method": "GridSearchCV (5-fold)",
        "Best Parameters": str(dt_grid.best_params_),
        "Best CV Accuracy": dt_grid.best_score_,
    },
    "Random Forest": {
        "Search Method": "GridSearchCV (5-fold)",
        "Best Parameters": str(rf_grid.best_params_),
        "Best CV Accuracy": rf_grid.best_score_,
    },
}).T
tuning_summary.to_csv(f"{RES_DIR}/hyperparameter_tuning_results.csv")
print("\n=== Hyperparameter Tuning Results Summary ===")
print(tuning_summary)


=== Hyperparameter Tuning Results Summary ===
                       Search Method  \
Decision Tree  GridSearchCV (5-fold)   
Random Forest  GridSearchCV (5-fold)   

                                                 Best Parameters  \
Decision Tree  {'criterion': 'gini', 'max_depth': 5, 'min_sam...   
Random Forest  {'bootstrap': True, 'max_depth': 10, 'max_feat...   

              Best CV Accuracy  
Decision Tree         0.938462  
Random Forest         0.964835  


## 9. 5-FOLD CROSS-VALIDATION PERFORMANCE COMPARISON (Table 3)

In [10]:
cv_dt = cross_validate(best_dt, X_train, y_train, cv=cv_strategy, scoring="accuracy")
cv_rf = cross_validate(best_rf, X_train, y_train, cv=cv_strategy, scoring="accuracy")

cv_table3 = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)] + ["Average"],
    "Decision Tree": list(cv_dt["test_score"]) + [cv_dt["test_score"].mean()],
    "Random Forest": list(cv_rf["test_score"]) + [cv_rf["test_score"].mean()],
}).set_index("Fold")
cv_table3.to_csv(f"{RES_DIR}/cv_accuracy_comparison.csv")
print("\n=== Table 3: 5-Fold Cross-Validation Accuracy Comparison ===")
print(cv_table3)

fig, ax = plt.subplots(figsize=(7, 5))
folds = np.arange(1, 6)
ax.plot(folds, cv_dt["test_score"], marker="o", label="Decision Tree", color="#c0392b")
ax.plot(folds, cv_rf["test_score"], marker="s", label="Random Forest", color="#2980b9")
ax.legend()
_bold_axis_labels(ax, "Fold", "Accuracy", "5-Fold Cross-Validation Accuracy Comparison")
_save_eps(fig, f"{FIG_DIR}/cv_accuracy_comparison.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Table 3: 5-Fold Cross-Validation Accuracy Comparison ===
         Decision Tree  Random Forest
Fold                                 
Fold 1        0.945055       0.967033
Fold 2        0.945055       0.978022
Fold 3        0.912088       0.934066
Fold 4        0.923077       0.956044
Fold 5        0.967033       0.989011
Average       0.938462       0.964835


## 10. EVALUATION METRICS: ROC CURVES (both models on test set)

In [11]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in [("Decision Tree (Tuned)", best_dt), ("Random Forest (Tuned)", best_rf)]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.8)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.legend()
_bold_axis_labels(ax, "False Positive Rate", "True Positive Rate", "ROC Curves")
_save_eps(fig, f"{FIG_DIR}/roc_curves.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 11. RANDOM FOREST: n_estimators vs ACCURACY (ensemble size effect)

In [12]:
n_est_values = [1, 5, 10, 25, 50, 100, 150, 200, 300]
n_est_rows = []
for n in n_est_values:
    model = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE)
    cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy, scoring="accuracy")
    n_est_rows.append({"n_estimators": n, "CV Accuracy": cv_res["test_score"].mean()})
n_est_df = pd.DataFrame(n_est_rows)
n_est_df.to_csv(f"{RES_DIR}/n_estimators_vs_accuracy.csv", index=False)
print("\n=== Random Forest: n_estimators vs CV Accuracy ===")
print(n_est_df)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(n_est_df["n_estimators"], n_est_df["CV Accuracy"], marker="o", color="#16a085")
_bold_axis_labels(ax, "Number of Trees (n_estimators)", "CV Accuracy",
                   "Random Forest: Ensemble Size vs Accuracy")
_save_eps(fig, f"{FIG_DIR}/n_estimators_vs_accuracy.eps")
plt.close(fig)


=== Random Forest: n_estimators vs CV Accuracy ===
   n_estimators  CV Accuracy
0             1     0.925275
1             5     0.949451
2            10     0.956044
3            25     0.956044
4            50     0.956044
5           100     0.962637
6           150     0.964835
7           200     0.962637
8           300     0.964835


## 12. FEATURE IMPORTANCE COMPARISON (Decision Tree vs Random Forest)

In [13]:
dt_importance = pd.Series(best_dt.feature_importances_, index=feature_names)
rf_importance = pd.Series(best_rf.feature_importances_, index=feature_names)
importance_df = pd.DataFrame({
    "Decision Tree": dt_importance,
    "Random Forest": rf_importance,
}).sort_values("Random Forest", ascending=False)
importance_df.to_csv(f"{RES_DIR}/feature_importance_comparison.csv")
print("\n=== Feature Importance Comparison (top 10) ===")
print(importance_df.head(10))

top10 = importance_df.head(10)
fig, ax = plt.subplots(figsize=(9, 6))
top10.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=11))
_bold_axis_labels(ax, "Importance", "Feature", "Feature Importance: Decision Tree vs Random Forest (Top 10)")
_save_eps(fig, f"{FIG_DIR}/feature_importance_comparison.eps")
plt.close(fig)


=== Feature Importance Comparison (top 10) ===
                      Decision Tree  Random Forest
worst concave points       0.123684       0.130588
worst area                 0.002056       0.118600
worst radius               0.751607       0.107072
mean perimeter             0.000000       0.086259
worst perimeter            0.000000       0.069519
mean radius                0.000000       0.065605
mean concave points        0.000000       0.063443
mean concavity             0.000000       0.061326
mean area                  0.000000       0.050850
area error                 0.007420       0.040344


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 13. COMPARATIVE SUMMARY TABLE

In [14]:
comparison_summary = pd.DataFrame([dt_metrics, rf_metrics]).set_index("Model")
comparison_summary.to_csv(f"{RES_DIR}/comparative_analysis.csv")
print("\n=== Comparative Analysis ===")
print(comparison_summary)

fig, ax = plt.subplots(figsize=(8, 5))
comparison_summary[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", ax=ax)
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=11))
_bold_axis_labels(ax, "Model", "Score", "Decision Tree vs Random Forest: Metric Comparison")
plt.xticks(rotation=15, ha="right")
_save_eps(fig, f"{FIG_DIR}/model_comparison_bar.eps")
plt.close(fig)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Comparative Analysis ===
                       Accuracy  Precision    Recall  F1-score   ROC-AUC  \
Model                                                                      
Decision Tree (Tuned)  0.903509   0.906077  0.903509  0.904146  0.935847   
Random Forest (Tuned)  0.956140   0.956073  0.956140  0.956027  0.992394   

                       Training Time (s)  
Model                                     
Decision Tree (Tuned)           0.005723  
Random Forest (Tuned)           0.138258  

All figures saved under: /home/claude/notebooks/figures
All result tables saved under: /home/claude/notebooks/results

Done.
